# Session 1 — Production Readiness Checklist

**Goal:** Evaluate how 'production-ready' our model actually is.

We'll run the test suite, check data quality, and work through a checklist that highlights what's still missing — setting up Session 2.

In [0]:
%run ../utils/config

In [0]:
%pip install ../bundle/wheels/ --quiet

## ✅ Check 1: Unit Tests

The `churn_model` package has a test suite in <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">bundle/wheels/tests/</span>.
Let's run it and see if our feature engineering code is correct.

In [0]:
import subprocess, sys, os

env = os.environ.copy()
env["PYTHONDONTWRITEBYTECODE"] = "1"

result = subprocess.run(
    [sys.executable, "-m", "pytest",
     "../bundle/wheels/tests/",
     "-v", "--tb=short", "--no-header", "-q",
     "--assert=plain",
     "-p", "no:cacheprovider"],
    capture_output=True,
    text=True,
    env=env,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    print("\n⚠️  Some tests failed. Review the output above.")
else:
    print("✓ All tests passed!")

## ✅ Check 2: Data Quality

Before trusting any model, we should validate the input data.
Our <span style="color:cc0000;background-color:#e0e0e0;font-family:courier">run_data_quality_checks()</span> function runs a set of lightweight checks.

_**Spoiler alert** These quality checks will be an integral part of our model pipeline in Session 2._

In [0]:
import yaml, pandas as pd
from churn_model.features import run_data_quality_checks

config_path = "../common/config.yml"
with open(config_path) as f:
    config = yaml.safe_load(f)

df = spark.table(f"{catalog}.`00_shared`.telco_churn").toPandas()
results = run_data_quality_checks(df, config)

print(f"{'Check':<40} {'Passed':>8} {'Actual Value':>20}")
print("-" * 72)
for _, row in results.iterrows():
    icon = "✓" if row["passed"] else "✗"
    print(f"{icon} {row['check_name']:<38} {str(row['passed']):>8} {row['actual_value']:>20}")

## Production Readiness Scorecard

Let's honestly evaluate where we are:

| Area | Status | Notes |
|---|---|---|
| ✅ Model tracked in MLflow | **Done** | Params, metrics, artifacts all logged |
| ✅ Model registered in Unity Catalog | **Done** | With `@champion` alias |
| ✅ Serving endpoint deployed | **Done** | REST API accepting requests |
| ✅ API governance (AI Gateway) | **Done** | Rate limiting + inference logging enabled |
| ✅ Unit tests passing | **Done** | Feature pipeline tested |
| ✅ Data quality checks | **Done** | Lightweight but effective |
| ✅ Feature store integration | **Done** | Features centralised in Unity Catalog; `fe.log_model()` enables lineage + skew prevention |
| ❌ Automated retraining | **Missing** | How does the model update when data changes? |
| ❌ Performance monitoring | **Missing** | How do we know if accuracy degrades? |
| ❌ Drift detection | **Missing** | What if input distributions shift? |
| ❌ Alerting | **Missing** | Who gets paged when something breaks? |
| ❌ Rollback mechanism | **Missing** | How do we recover from a bad deployment? |

**Session 2** will address the remaining ❌'s

## Discussion Questions

Share with the group:
1. **How long does it take your team to go from 'trained model' to 'monitored production endpoint'?**
2. **What failure modes have you seen in production ML deployments?**
3. **Who owns the model once it's deployed — the data scientist or the engineering team?**

➡️ Break — then Session 2: `../session2/01_cicd_overview.ipynb`